# TDWI Lab 3 Part 3: PR Review Automations

In this lesson you will configure a **Cursor Automation** that reviews pull requests after a human marks them ready, then run a **Cloud Agent** to add a Streamlit **Revenue Explorer** on top of the cleaned sales pipeline. The automation fires when you click **Ready for review** on GitHub—not when the agent first opens a draft PR.

## Learning Objectives

By the end of this mini-lesson you will be able to:
- Create a GitHub PR-triggered Cursor Automation
- Use the **Pull request opened** trigger (including when a draft is marked ready for review)
- Configure **Comment on pull request** as the automation output
- Run a Cloud Agent to add a Streamlit Revenue Explorer and observe the automation review on a ready PR
- Contrast a custom review automation with Bugbot (optional on your repo)

## Prerequisites

- Completed [README.md](README.md) setup (fork, clone, local `.venv`, test push)
- Completed [LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb](LAB3-Part-1-Cloud-Agent-Environment-Setup.ipynb) (env/secrets on your fork)
- Completed [LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb](LAB3-Part-2-Running-Cursor-Cloud-Agents.ipynb): pipeline fixes merged to `main`, tests green
- Cursor plan with **Automations** enabled; GitHub connected to **your fork**

**Important:** Configure the automation on the **same GitHub repo where you open PRs** (your fork). Each student sets up their own automation, unless the instructor demos on a shared fork.

## Step 1: Understand the workflow

Lab 3 uses a deliberate handoff between humans and agents:

1. **Cloud Agent** implements code and opens a **draft PR**.
2. **You** pull the branch, run tests locally, and review the diff.
3. **You** click **Ready for review** on GitHub when you want automated feedback.
4. **Cursor Automation** posts review comments on the PR.

Cursor defines two related triggers ([Automations docs](https://cursor.com/docs/cloud-agent/automations)):

| Trigger | When it fires |
|---------|----------------|
| **Draft opened** | A draft PR is created |
| **Pull request opened** | A non-draft PR is created **or** a draft is **marked ready for review** |

For this lab, use **Pull request opened** so the review runs only after your human review step—not when the agent first creates a draft.

## Step 2: Create the PR review automation

1. Open Cursor **Settings** → **Automations** (or the Automations section on [cursor.com](https://cursor.com)). See the [Automations documentation](https://cursor.com/docs/cloud-agent/automations) if the UI differs slightly.
2. Click **Create automation** (or equivalent).
3. **Trigger:** GitHub → **Pull request opened**.
4. **Repository:** Select **your fork** of this starter repo (e.g. `your-username/tdwi-agentic-sales-pipeline-starter`).
5. **Tools / output:** Enable **Comment on pull request** (and any repo access the UI requires for reading the diff).
6. Paste the following into the automation **instructions** field:

```text
Review this pull request for the TDWI sales pipeline lab.

Focus on:
- Whether cleaning logic is reused from generate_sales_report.load_and_clean_data() (not duplicated)
- Whether requirements.txt pins any new dependencies (e.g. streamlit)
- Whether existing tests in test_sales_report.py would still pass; note if the PR does not mention test results
- Filter/UI edge cases if a Streamlit app was added
- Scope: flag unrelated refactors

Post a concise review as PR comments: summary, strengths, 1–3 suggestions. Do not merge or approve.
```

7. Save the automation.

## Step 3: Save and verify the automation

- Confirm the automation appears in your Automations list and is enabled for your fork.
- Use the dashboard **run history** (if available) after Step 6 to confirm it executed.

**Note:** If you already marked a Part 2 PR as ready, toggling ready again may not re-fire the trigger. Part 3’s new PR (Step 4) is the intended test.

## Step 4: Cloud Agent — add Revenue Explorer

Start a Cloud Agent on **your fork** (same Dockerfile-managed environment as Part 1). Go to [cursor.com/agents](https://cursor.com/agents) or the Agents window in Cursor, select your repository, and paste this prompt:

```text
Add a small Streamlit app called revenue_explorer.py at the repo root.

Requirements:
- Import and use load_and_clean_data() from generate_sales_report.py — do not duplicate cleaning logic.
- Sidebar: date range filter, multi-select product, optional customer_id filter.
- Main area: KPIs (total revenue, order count, average order value) for the filtered data.
- One chart: daily revenue trend for the filtered data.
- Add streamlit to requirements.txt with a pinned version.
- Add a one-line comment at the top: streamlit run revenue_explorer.py
- Run python -m pytest test_sales_report.py and ensure tests still pass.
- Open a PR when done. Do not modify README.md or any lab notebook.
```

Wait for the agent to finish. It will typically open a **draft PR**.

## Step 5: Human review (same pattern as Part 2)

1. Open the draft PR on GitHub and note the branch name.
2. Locally, fetch and check out the agent branch:

```bash
git fetch origin
git checkout <agent-branch-name>
```

3. With `.venv` activated, run tests:

```bash
python -m pytest test_sales_report.py
```

4. Optional: run the new app locally (install deps if the agent added `streamlit`):

```bash
pip install -r requirements.txt
streamlit run revenue_explorer.py
```

5. Read the code changes. Do **not** merge yet—you will trigger the automation in the next step.

## Step 6: Mark the PR ready for review

1. On the GitHub PR page, click **Ready for review** (draft PRs only).
2. This event counts as **Pull request opened** for Cursor Automations—your review automation should start within a few minutes.
3. Refresh the PR **Conversation** tab and read the automation’s comments.
4. If Bugbot is enabled on your repo, compare its findings with your custom automation. They serve different purposes: Bugbot is product-default; your automation follows **your** checklist.
5. Optionally merge after you are satisfied with the review and your local testing.

## Step 7: Debrief questions

1. Why use a **draft PR** for the agent and **ready for review** for the automation?
2. What did your automation catch that you would have missed? What did it miss?
3. When would you use **Automations** vs **Bugbot** vs both?
4. How could you add a **CI completed** trigger so review runs only after green checks?

## Instructor checklist

- Confirm students configure Automations on **their fork**, not only the upstream template repo.
- Confirm Automations is available on student Cursor plans before class.
- Demo flow: draft PR → local pytest → **Ready for review** → automation comments appear.
- Timebox: ~20–30 minutes for automation setup + agent run + review.